In [1]:
!hostname

psh01com1hcom41


In [24]:
import os, sys
from pathlib import Path
import muon as mu
import pandas as pd
import numpy as np
import logging
from matplotlib_inline import backend_inline
import matplotlib.pyplot as plt

os.environ["PATH"] = f"{Path(sys.executable).parent}:{os.environ['PATH']}"

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
sys.path.append(str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR.parent / "data"

import mudata
mudata.set_options(pull_on_update=False)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

backend_inline.set_matplotlib_formats('svg')

organism_code = "hg38"

data_dir = PROJECT_DIR / "data"

gene_tss_file = data_dir / "genome_data" / "genome_annotation" / organism_code / "gene_tss.bed"
organism_tad_file = data_dir / "genome_data" / "genome_annotation" / organism_code / "hg38_TADs.bed"

dataset_name = "ENCODE_adrenal"
sample_name = "adrenal_adult_ENCDO221IPH"
cell_type = "Fibroblast"

processed_data_dir = data_dir / "processed" / organism_code / dataset_name / sample_name / "multiome_processed.h5mu"


In [ ]:

mdata = mudata.read_h5mu(processed_data_dir)

mdata = mdata[mdata["rna"].obs["celltype"] == cell_type].copy()

rna = mdata.mod['rna']
atac = mdata.mod['atac']

genes = rna.var_names
peaks = atac.var_names

In [36]:
def _assign_tad(chrom, pos, tads):
    """TAD id ("chr|i") of each point; "" for a point that falls in no TAD."""
    # Empty array for each position
    out = np.full(len(pos), "", dtype=object)

    # Group each TAD by chromosome
    for chr, tad in tads.groupby("chrom", sort=False):

        # Get the indicies of the genes on this chromosome
        idx = np.flatnonzero(chrom == chr)
        if idx.size == 0:
            continue

        # Get the sorted start, end, and idx TADs for this chromosome
        tad = tad.sort_values("start")
        starts, ends, ids = tad["start"].to_numpy(), tad["end"].to_numpy(), tad["tad"].to_numpy()

        # Find the TAD that each gene is in
        i = np.searchsorted(starts, pos[idx], side="right") - 1
        ok = (i >= 0) & (pos[idx] < ends[i.clip(0)])

        # Assign the TAD id to the output array for each gene that is in a TAD
        # example: chr1|1956
        out[idx[ok]] = ids[i[ok]]
    
    return out


def peak_nearest_tss_genes(peak_names, tss_bed, tad_bed=None):
    """Nearest-TSS gene of every ATAC peak, optionally only within the peak's TAD.

    Without tad_bed: nearest TSS genome-wide (bedtools closest -d -t first).
    With tad_bed: nearest TSS among the TSSs in the same TAD as the peak midpoint.
    A peak outside every TAD, or in a TAD with no TSS, gets "".
    Returns upper-case gene names in peak column order. Needs bedtools on PATH.
    """
    import shutil, subprocess, tempfile

    tss_bed = Path(tss_bed)
    if not tss_bed.is_file():
        raise FileNotFoundError(f"TSS annotation not found: {tss_bed}")
    if shutil.which("bedtools") is None:
        raise RuntimeError("bedtools is not on PATH (activate my_env)")

    # chrom:start-end -> chrom, start, end, col
    peak_names = np.asarray(peak_names).astype(str)
    coords = pd.Series(peak_names).str.extract(r"^(chr[^:]+):(\d+)-(\d+)$")
    if coords.isna().any().any():
        raise ValueError(f"Peak names must be chrom:start-end; first is {peak_names[0]!r}")
    peaks = pd.DataFrame({
        "chrom": coords[0].to_numpy(), "start": coords[1].astype(np.int64).to_numpy(),
        "end": coords[2].astype(np.int64).to_numpy(), "col": np.arange(len(peak_names)),
    })

    # Read in the gene TSS locations -> chrom, start, end, gene
    tss = pd.read_csv(tss_bed, sep="\t", header=None, usecols=[0, 1, 2, 3],
                      names=["chrom", "start", "end", "gene"], dtype={"gene": str})

    if tad_bed is not None:
        # The TAD file is space-separated, so read it with a whitespace separator.
        tads = pd.read_csv(tad_bed, sep=r"\s+", header=None, usecols=[0, 1, 2],
                           names=["chrom", "start", "end"])

        # Format the TAD id as "chrom|i" where i is the TAD index in the TAD file. This is used to
        # replace the chromosome in the peaks and TSS locations, so that bedtools closest only
        # searches for TSSs in the same TAD as the peak.
        tads["tad"] = tads["chrom"] + "|" + tads.index.astype(str)

        # Replace the chromosome with the TAD id. bedtools closest only looks
        # for a TSS on the same "chromosome", so it only searches inside the TAD.
        peak_centers = (peaks["start"] + peaks["end"]) // 2
        peaks["chrom"] = _assign_tad(peaks["chrom"].to_numpy(),
                                     peak_centers.to_numpy(), tads)
        tss["chrom"] = _assign_tad(tss["chrom"].to_numpy(), tss["start"].to_numpy(), tads)

        # Remove peaks or TSS locations that are outside of every TAD
        peaks = peaks[peaks["chrom"] != ""]
        tss = tss[tss["chrom"] != ""]

        # Keep only TSSs in TADs that have a peak. bedtools closest stops with an
        # error when the TSS file has a "chromosome" that the peak file does not.
        tss = tss[tss["chrom"].isin(set(peaks["chrom"]))]

    
    with tempfile.TemporaryDirectory() as tmpdir:
        # Use temporary tiles of the prepared peaks and TSS locations for bedtools
        tmp = Path(tmpdir)
        peaks.to_csv(tmp / "peaks.bed", sep="\t", header=False, index=False)
        tss.to_csv(tmp / "tss_in.bed", sep="\t", header=False, index=False)

        # Same sort as build_celltype_ground_truth.py: with -t first, the TSS order
        # decides ties between TSSs at the same distance.
        subprocess.run(
            f"set -o pipefail; export LC_ALL=C; "
            f"sort -k1,1 -k2,2n {tmp}/tss_in.bed > {tmp}/tss.bed && "
            f"sort -k1,1 -k2,2n {tmp}/peaks.bed > {tmp}/peaks.sorted.bed && "
            f"bedtools closest -a {tmp}/peaks.sorted.bed -b {tmp}/tss.bed -d -t first "
            f"| cut -f4,8 > {tmp}/nearest.tsv",
            shell=True, check=True, executable="/bin/bash",
        )
        nearest = pd.read_csv(tmp / "nearest.tsv", sep="\t", header=None,
                              names=["col", "gene"], dtype={"gene": str}, na_filter=False)

    # Map the nearest TSS to each peak
    # If no TSS is found within the TAD, the gene will be an empty string.
    genes = np.full(len(peak_names), "", dtype=object)
    found = nearest["gene"].ne(".").to_numpy()
    genes[nearest["col"].to_numpy()[found]] = nearest["gene"].str.upper().to_numpy()[found]
    return genes

In [38]:
gene_to_peaks = peak_nearest_tss_genes(peaks, gene_tss_file, tad_bed=organism_tad_file)
gene_to_peaks_df = pd.DataFrame({"peak": peaks, "gene": gene_to_peaks})
gene_to_peaks_df

,peak,gene
0,chr1:794909-795641,
1,chr1:810897-811712,FAM87B
2,chr1:816802-818420,FAM87B
3,chr1:818700-818898,FAM87B
4,chr1:826724-828070,LINC01128
...,...,...
129586,chrY:25639760-25640287,
129587,chrY:25643119-25644035,
129588,chrY:25821759-25822256,
129589,chrY:25851004-25851218,
